# Jaggery Quality & FSSAI Non-Compliance — ML Workflow

**Dataset:** `FSSAI_Jaggery_3060_ML_Dataset.csv`  
**Task:** Explore whether available sample descriptors can estimate the reconstructed non-compliance label.

> **Scientific limitation — read before interpreting results:** This is an *aggregate-derived reconstruction*, not FSSAI's original sample-level laboratory dataset. Individual sample rows and their joint relationships were not published. Several row-level fields (including individual labels and issue assignments) were reconstructed from aggregate totals. Therefore, model metrics are a software/pipeline demonstration only. They are **not evidence of real-world predictive performance** and must not be reported as validated FSSAI model accuracy.

The notebook deliberately excludes fields that encode or directly derive from the target, such as `Primary_Quality_Issue`, `High_Moisture`, `High_Sulphite`, `Added_Colour`, `Other_Quality_NC`, and `Quality_Substandard_1110` when predicting `Non_Compliant`.


## 1. Install/import dependencies and load the data

Place the CSV in the same folder as this notebook. In Google Colab, upload the CSV when prompted.

In [ ]:
# If needed, uncomment:
# %pip install pandas numpy matplotlib seaborn scikit-learn joblib

from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GroupShuffleSplit, GroupKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, ConfusionMatrixDisplay,
    confusion_matrix, balanced_accuracy_score
)
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance
import joblib

# Find CSV beside notebook; otherwise allow upload in Colab.
csv_path = Path("FSSAI_Jaggery_3060_ML_Dataset.csv")
if not csv_path.exists():
    try:
        from google.colab import files
        uploaded = files.upload()
        csv_path = Path(next(iter(uploaded.keys())))
    except Exception:
        raise FileNotFoundError(
            "Place FSSAI_Jaggery_3060_ML_Dataset.csv beside this notebook "
            "or upload it in Google Colab."
        )

df = pd.read_csv(csv_path)
print("Shape:", df.shape)
display(df.head())


## 2. Data audit

Check dimensions, missing values, duplicates, class balance, and whether the published aggregate totals were preserved. A row count of 3,060 does not make these 3,060 independently observed laboratory records.

In [ ]:
print("Rows, columns:", df.shape)
print("\\nMissing values:")
display(df.isna().sum().sort_values(ascending=False).to_frame("missing_count"))
print("\\nDuplicate rows:", df.duplicated().sum())

target = "Non_Compliant"
if target not in df.columns:
    raise ValueError(f"Target column {target!r} not found.")

print("\\nTarget counts:")
display(df[target].value_counts(dropna=False).rename("count").to_frame())
print("\\nTarget proportions:")
display(df[target].value_counts(normalize=True).rename("proportion").to_frame())

assert len(df) == 3060, "Unexpected row count; verify the input file."
print("Non_Compliant positives:", int(df[target].sum()))


## 3. Exploratory data analysis

These plots describe the reconstructed file. They should not be interpreted as fresh findings from individual FSSAI samples.

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x=target)
plt.title("Reconstructed non-compliance label distribution")
plt.xlabel("Non-compliant (1) / compliant (0)")
plt.ylabel("Number of rows")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
state_rate = df.groupby("State")[target].mean().sort_values(ascending=False)
state_rate.plot(kind="bar", ax=ax[0])
ax[0].set_title("Reconstructed non-compliance rate by state")
ax[0].set_ylabel("Rate")
ax[0].tick_params(axis="x", rotation=90)

type_rate = df.groupby("Jaggery_Type")[target].mean().sort_values(ascending=False)
type_rate.plot(kind="bar", ax=ax[1])
ax[1].set_title("Reconstructed non-compliance rate by jaggery type")
ax[1].set_ylabel("Rate")
ax[1].tick_params(axis="x", rotation=35)
plt.tight_layout()
plt.show()


## 4. Define features and prevent target leakage

We use only descriptors intended to be available before laboratory testing. We exclude all target-derived fields, issue flags, sample IDs, source/provenance columns, and the alternative substandard target.

**Important:** `State` is included for a within-known-state demonstration. Because the reconstructed labels were allocated to match state-level totals, this feature can encode those aggregate rates. We therefore also run a stricter state-held-out evaluation below.

In [ ]:
feature_cols = [
    "State", "Region", "Jaggery_Type", "Form",
    "Packed", "Ecommerce", "Collection_Channel"
]
X = df[feature_cols].copy()
y = df[target].astype(int).copy()
groups = df["State"].astype(str).copy()

# Treat category-like numeric flags as categorical.
cat_cols = ["State", "Region", "Jaggery_Type", "Form", "Collection_Channel",
            "Packed", "Ecommerce"]
num_cols = []

print("Features:", feature_cols)
print("Excluded target-derived columns:", [
    c for c in df.columns
    if c in ["Primary_Quality_Issue", "High_Moisture", "High_Sulphite",
             "Added_Colour", "Other_Quality_NC", "Quality_Substandard_1110"]
])


## 5. Preprocessing and models

All preprocessing is fitted inside each pipeline to avoid fitting encoders on the test data. We compare a majority-class baseline with several standard classifiers.

In [ ]:
categorical_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer(
    transformers=[("cat", categorical_pipe, cat_cols)],
    remainder="drop"
)

models = {
    "Dummy baseline": DummyClassifier(strategy="most_frequent"),
    "Logistic Regression": LogisticRegression(max_iter=2000, class_weight="balanced"),
    "Decision Tree": DecisionTreeClassifier(
        max_depth=5, min_samples_leaf=20, class_weight="balanced", random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=400, min_samples_leaf=10, class_weight="balanced_subsample",
        random_state=42, n_jobs=-1
    ),
    "Extra Trees": ExtraTreesClassifier(
        n_estimators=400, min_samples_leaf=10, class_weight="balanced",
        random_state=42, n_jobs=-1
    )
}

pipelines = {
    name: Pipeline([("prep", preprocess), ("model", model)])
    for name, model in models.items()
}


## 6. Evaluation A — random stratified split

This estimates performance when rows from the same states can appear in both train and test sets. Since state-wise target counts were preserved during reconstruction, this can reward learning state-level prevalence. Treat this as a pipeline check, not independent validation.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

def evaluate_models(model_dict, X_train, X_test, y_train, y_test):
    rows = []
    fitted = {}
    for name, pipe in model_dict.items():
        pipe.fit(X_train, y_train)
        pred = pipe.predict(X_test)
        proba = pipe.predict_proba(X_test)[:, 1] if hasattr(pipe, "predict_proba") else None
        rows.append({
            "Model": name,
            "Accuracy": accuracy_score(y_test, pred),
            "Balanced_Accuracy": balanced_accuracy_score(y_test, pred),
            "Precision": precision_score(y_test, pred, zero_division=0),
            "Recall": recall_score(y_test, pred, zero_division=0),
            "F1": f1_score(y_test, pred, zero_division=0),
            "ROC_AUC": roc_auc_score(y_test, proba) if proba is not None and y_test.nunique() == 2 else np.nan
        })
        fitted[name] = pipe
    return pd.DataFrame(rows).sort_values("F1", ascending=False), fitted

random_results, random_fitted = evaluate_models(
    pipelines, X_train, X_test, y_train, y_test
)
display(random_results.round(3))


## 7. Evaluation B — hold out entire states

This is the more demanding test: no state in the test set appears in training. It better tests whether patterns transfer to unseen states. With only aggregate-reconstructed data, it still cannot establish real-world validity.

In [ ]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

Xg_train, Xg_test = X.iloc[train_idx], X.iloc[test_idx]
yg_train, yg_test = y.iloc[train_idx], y.iloc[test_idx]

print("Train states:", Xg_train["State"].nunique())
print("Test states:", Xg_test["State"].nunique())
print("Overlapping states:", len(set(Xg_train.State) & set(Xg_test.State)))

group_results, group_fitted = evaluate_models(
    pipelines, Xg_train, Xg_test, yg_train, yg_test
)
display(group_results.round(3))


## 8. Inspect the selected model

Select the best model by F1 from the state-held-out evaluation. If performance is weak, that is an informative result: the available descriptors may not generalize, and the reconstructed labels do not contain true sample-level signals.

In [ ]:
best_name = group_results.iloc[0]["Model"]
best_model = group_fitted[best_name]
best_pred = best_model.predict(Xg_test)

print("Selected model:", best_name)
print(classification_report(yg_test, best_pred, zero_division=0))

ConfusionMatrixDisplay.from_predictions(yg_test, best_pred)
plt.title(f"State-held-out confusion matrix — {best_name}")
plt.tight_layout()
plt.show()


## 9. Feature importance (permutation)

Permutation importance is calculated on the held-out states. Interpret it cautiously: state and region are correlated, and the underlying row-level associations were reconstructed.

In [ ]:
perm = permutation_importance(
    best_model, Xg_test, yg_test, n_repeats=10,
    random_state=42, scoring="f1"
)
importance = pd.DataFrame({
    "Feature": feature_cols,
    "Importance_Mean": perm.importances_mean,
    "Importance_STD": perm.importances_std
}).sort_values("Importance_Mean", ascending=False)

display(importance.round(4))
sns.barplot(data=importance, x="Importance_Mean", y="Feature")
plt.title("Permutation importance on state-held-out test data")
plt.tight_layout()
plt.show()


## 10. Save model and evaluation outputs

The saved model is a demonstration artifact trained on reconstructed records. Keep the README with it and label it accordingly.

In [ ]:
# Refit selected model on the full reconstructed dataset for a demonstration artifact.
# This is NOT a model validated on original FSSAI sample-level data.
final_demo_model = pipelines[best_name]
final_demo_model.fit(X, y)

joblib.dump(final_demo_model, "jaggery_noncompliance_demo_model.joblib")
random_results.to_csv("random_split_model_metrics.csv", index=False)
group_results.to_csv("state_heldout_model_metrics.csv", index=False)
importance.to_csv("permutation_feature_importance.csv", index=False)

print("Saved:")
print("- jaggery_noncompliance_demo_model.joblib")
print("- random_split_model_metrics.csv")
print("- state_heldout_model_metrics.csv")
print("- permutation_feature_importance.csv")


## 11. Final interpretation and next steps

### What this project can honestly demonstrate
- Data loading, validation, EDA and preprocessing
- Classification pipeline implementation
- Comparison against a simple baseline
- Random-split versus group-held-out evaluation
- Confusion matrix and permutation-importance workflow

### What it cannot establish
- Real-world jaggery adulteration detection
- Chemical quality prediction from moisture/sucrose/ash values
- Validated FSSAI sample-level compliance prediction
- Reliable generalization to new samples

### How to turn this into a scientifically strong project
Obtain the original sample-level FSSAI laboratory records (or collect a new laboratory dataset) with actual sample measurements and labels. Useful fields include moisture, sucrose, reducing sugars, total sugars, ash, sulphite, added colour, heavy metals, jaggery type/form, location and the official compliance outcome. Then define the compliance label using the applicable FSSAI standard and evaluate on a genuinely independent test set.

**Suggested honest project title:**  
*Machine Learning Workflow for Jaggery Quality Risk Assessment Using FSSAI Aggregate-Derived Data*
